# 01 · 시드 이미지 데이터 탐색 — 멀티모달 추출(이미지→JSON)

**TL;DR** — 영수증 이미지→구조화 JSON 시드 데이터셋(cord-v2)을 로드해 이미지와 정답 JSON을 눈으로 확인합니다.

**Why** — 멀티모달 학습은 '이미지→타깃 텍스트' 쌍이 필요합니다. 텍스트 트랙과 달리 합성 생성 단계 없이 공개 라벨 데이터를 직접 씁니다(이미지 합성은 별개 문제).

**기존 Pain Point** — 이미지 태스크는 permissive 라이선스 데이터가 드뭅니다. cord-v2(cc-by-4.0, ungated)는 영수증→JSON 라벨이 갖춰져 있어 이미지→구조화 추출 데모에 적합합니다.

> 🔴 실제 실행 시 AWS 자격증명·GPU·엔드포인트 과금이 발생합니다. 먼저 `DRY_RUN=1`로 파이프라인을 검증하세요.

In [ ]:
import os, sys
# 리포 루트를 path에 추가해 common/ 와 트랙 로컬 모듈을 import
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, REPO)
sys.path.insert(0, os.getcwd())

## 시드 데이터셋: naver-clova-ix/cord-v2
- **라이선스**: cc-by-4.0, ungated.
- **원본 포맷**: `image`(영수증 PNG) + `ground_truth`(문자열 JSON, `gt_parse.menu[]` 등).
- **이 트랙의 파싱**(`track_data.py`): 이미지 → 별도 `images` 컬럼(TRL VLM collator가 이미지 자리표시자를 주입), `gt_parse.menu` → `{menu:[{name,count,price}]}` 타깃 JSON.
- **성공 기준**: 영수증 이미지에서 메뉴/수량/가격을 정확한 JSON으로 추출.

**원본 row 예시**:
```text
image:        <영수증 PNG, 예 864x1296>
ground_truth: {"gt_parse": {"menu": [{"nm":"Nasi Campur Bali","cnt":"1 x","price":"75,000"}, ...]}}
```
→ 파싱 후 타깃: `{"menu": [{"name":"Nasi Campur Bali","count":"1 x","price":"75,000"}, ...]}`

In [ ]:
import importlib, track_data as td; importlib.reload(td)
from common import config
seeds = td.load_seed_examples(3, token=config.get_hf_token())
ex = seeds[0]
# TRL VLM 포맷: images 컬럼(별도) + messages(텍스트만). collator가 이미지 placeholder를 주입.
print('columns     :', [k for k in ex if not k.startswith('_')])
print('user text   :', ex['messages'][0]['content'][:120])
print('target JSON :', ex['messages'][1]['content'][:300])
print('num images  :', len(ex['images']))

In [ ]:
# 이미지 미리보기 (노트북에서 렌더)
img = ex['_image']   # load_seed_examples가 편의로 담아 준 원본 PIL
print('image size:', img.size)
img   # Jupyter가 이미지를 렌더링

✅ 이미지와 타깃 JSON을 확인했습니다. 다음은 **02_train_mm_sagemaker.ipynb**로 멀티모달 SFT를 실행합니다. (이 트랙은 합성 데이터 단계가 없습니다.)